# GPT scoring evaluation release copy

This is a cleaned copy of the supplied scoring implementation. It reads precomputed answers; it does not generate answers or establish their model identities or test-split origin. Stored outputs and private Drive paths were removed. The scoring and averaging functions are retained unchanged for auditability.

Set `CAEE_EVALUATION_CSV` to an authorized local comparison CSV. Results are written locally under `CAEE_EVALUATION_OUTPUT_DIR` (default: `evaluation_outputs`). Results may contain the input records and must not be published without a separate privacy review. Read `docs/REPRODUCIBILITY.md` for known issues. The full notebook has not been execution-tested during preparation.

This notebook sends questions, reference answers, and generated answers to an external API and may incur charges. Only use records that may lawfully and appropriately be sent for this purpose. Set `CAEE_ALLOW_EXTERNAL_EVALUATION=1` explicitly and provide `OPENAI_API_KEY` securely to enable execution. No key or participant data is included. The original fixed-denominator parse-failure behavior is retained and must be checked against historical run records before relying on the paper's scores.


In [ ]:
!pip install -qqq -U pandas==2.2.2 transformers datasets accelerate trl bitsandbytes tqdm huggingface_hub pyarrow==16.1.0 sentence-transformers openai

import gc
import os
import re
import torch
import transformers
import datasets
import shutil
import pandas as pd
import nltk
import openai
from openai import OpenAI
from tqdm import tqdm
from datasets import Dataset, Value, Features, Sequence, load_dataset
from huggingface_hub import login, HfApi
from sklearn.model_selection import train_test_split
from trl import ORPOConfig, ORPOTrainer, setup_chat_format
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig, pipeline, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util

# Credentials and local paths are supplied by the execution environment.
from pathlib import Path
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
evaluation_csv = os.environ.get("CAEE_EVALUATION_CSV")
if not evaluation_csv:
    raise ValueError("Set CAEE_EVALUATION_CSV to an authorized local comparison CSV.")
evaluation_output_dir = Path(os.environ.get("CAEE_EVALUATION_OUTPUT_DIR", "evaluation_outputs"))
evaluation_output_dir.mkdir(parents=True, exist_ok=True)

# External scoring is disabled unless explicitly enabled for authorized input.
if os.environ.get("CAEE_ALLOW_EXTERNAL_EVALUATION") != "1":
    raise RuntimeError("External scoring is disabled. Review data permissions and costs before setting CAEE_ALLOW_EXTERNAL_EVALUATION=1.")
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Provide OPENAI_API_KEY securely through the environment.")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


In [ ]:
# Load the comparison dataset
num_rows_selected = 256
results_df = pd.read_csv(evaluation_csv).sample(num_rows_selected, random_state=42)

# Combined prompt for evaluating CV descriptions on coherence, consistency, fluency, and relevance
combined_prompt = """
You are going to evaluate a question and two answers: an expected answer and a generated answer. Please evaluate the following four aspects of the generated answer compared to the expected answer:

1. Coherence: Does the generated answer logically flow, and are the ideas well-structured and connected?
2. Consistency: Does the generated answer align with the expected answer without contradicting itself?
3. Fluency: Is the generated answer grammatically correct, well-written, and easy to understand?
4. Relevance: Does the generated answer address the question effectively, and is it closely related to the expected answer?

Rate each aspect on a scale of 1 to 5, where 1 is very poor and 5 is excellent.
Please respond with the ratings for coherence, consistency, fluency, and relevance within brackets, like this: [coherence: 5, consistency: 4, fluency: 5, relevance: 3]
"""

# Helper function to extract the scores from the response
# We expect a response like: [coherence: 5, consistency: 4, fluency: 5, relevance: 3]
def extract_ratings(response_text):
    try:
        matches = re.findall(r'coherence: (\d+), consistency: (\d+), fluency: (\d+), relevance: (\d+)', response_text)
        if matches:
            coherence, consistency, fluency, relevance = map(int, matches[0])
            return coherence, consistency, fluency, relevance
    except (IndexError, ValueError):
        pass
    return None, None, None, None

# Direct-rating evaluation using the historical GPT-3.5 model identifier
def g_eval_gpt(prompt, question, expected_answer, generated_answer):
    attempt = 0
    while attempt < 3:  # Retry up to 3 times if parsing fails
        response = client.chat.completions.create(
            model="gpt-3.5-turbo-0125",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Question: {question}\nExpected Answer: {expected_answer}\nGenerated Answer: {generated_answer}"}
            ],
            max_tokens=128,
        )
        response_text = response.choices[0].message.content
        coherence, consistency, fluency, relevance = extract_ratings(response_text)
        if coherence is not None:
            return coherence, consistency, fluency, relevance
        attempt += 1

    return "unknown", "unknown", "unknown", "unknown"  # Return "unknown" if parsing fails after 3 attempts

# Function to evaluate generated answers on four aspects using GPT-3.5
def evaluate_answers(input_df, num_runs=5):
    for suffix in ['base', '4096', '4096x1-25', '4096x1-50', '4096x1-75']:
        coherence_column = f'{suffix}_coherence_gpt'
        consistency_column = f'{suffix}_consistency_gpt'
        fluency_column = f'{suffix}_fluency_gpt'
        relevance_column = f'{suffix}_relevance_gpt'

        # Initialize columns to 0 so we can calculate the average of N runs
        input_df[coherence_column] = 0.0
        input_df[consistency_column] = 0.0
        input_df[fluency_column] = 0.0
        input_df[relevance_column] = 0.0

        # Loop N times and accumulate the results
        for run in range(num_runs):
            print(f"Running evaluation for {suffix}, iteration {run+1}/{num_runs}")
            for idx, row in tqdm(input_df.iterrows(), total=len(input_df), desc=f"Evaluating dimensions ({suffix}) using GPT-3.5"):
                question = row['question']
                expected_answer = row['expected_answer']
                generated_answer = row[f'parsed_answer_{suffix}']

                # GPT-3.5 evaluation
                coherence, consistency, fluency, relevance = g_eval_gpt(combined_prompt, question, expected_answer, generated_answer)

                # Accumulate the scores for averaging
                if coherence != "unknown":
                    input_df.at[idx, coherence_column] += coherence / num_runs
                    input_df.at[idx, consistency_column] += consistency / num_runs
                    input_df.at[idx, fluency_column] += fluency / num_runs
                    input_df.at[idx, relevance_column] += relevance / num_runs

    return input_df

# Perform evaluation
results_df = evaluate_answers(results_df)

# Summary of results
summary = []

metrics = ['coherence', 'consistency', 'fluency', 'relevance']
for metric in metrics:
    gpt_avg_scores = []

    for suffix in ['base', '4096', '4096x1-25', '4096x1-50', '4096x1-75']:
        gpt_column = f'{suffix}_{metric}_gpt'
        gpt_avg = pd.to_numeric(results_df[gpt_column], errors='coerce').mean()
        gpt_avg_scores.append(gpt_avg)

    summary.append({
        'metric': metric,
        'gpt_avg_base': gpt_avg_scores[0],
        'gpt_avg_4096': gpt_avg_scores[1],
        'gpt_avg_4096x1-25': gpt_avg_scores[2],
        'gpt_avg_4096x1-50': gpt_avg_scores[3],
        'gpt_avg_4096x1-75': gpt_avg_scores[4],
    })

# Adding an average row for all metrics
average_scores = {
    'metric': 'average',
}
for suffix in ['base', '4096', '4096x1-25', '4096x1-50', '4096x1-75']:
    average_scores[f'gpt_avg_{suffix}'] = sum([summary_item[f'gpt_avg_{suffix}'] for summary_item in summary]) / len(metrics)

summary.append(average_scores)

summary_df = pd.DataFrame(summary)

# Display the comparison summary
print(summary_df)

# Save the updated DataFrame with the final results
results_df.to_csv(evaluation_output_dir / f'inference_comparison_with_geval_metrics_gpt_{num_rows_selected}.csv', index=False)
summary_df.to_csv(evaluation_output_dir / f'comparison_summary_geval_metrics_gpt_{num_rows_selected}.csv', index=False)

# Clean up memory before ending the script
gc.collect()
torch.cuda.empty_cache()
